# Task 1.3 - Augmentation Analysis (what does `*_augmented` do?)

Condensed re-confirmation of the provided shift in `data/calibration_augmented/` (cross-checked on `data/validation_augmented/`). The `*_augmented` splits are byte-for-byte the same data the old k=16 attempt analysed (see `notebooks/old/old_04_task13_aug_analysis.ipynb`), so this notebook re-runs the core measurements only - the battery, the direct JPEG-quality tables, and the per-image distribution - and regenerates the report figures. Conclusions are ported to `task13_experiment_log.md`. Observational and from **unpaired** splits: we identify the transform *family*, its per-image application, and rough magnitude, not the exact generator.

## 0 - Setup

Decode each split two ways: (a) **raw** PIL metadata (size, aspect, mode, encoded byte length) *before* any cleaning, so geometric/format/compression changes are not hidden by our shorter-edge-resize+center-crop; and (b) the **cleaned** 224x224 uint8 array used downstream. Cached to `notebook_cache/` after first run.

In [ ]:
import os, sys, io as _stdio, json
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
sys.path.insert(0, "../solution")

%matplotlib inline
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

from _lib import io as _io

DATA    = Path("../solution/data")
NBCACHE = Path("../solution/artifacts/notebook_cache")
FIGS    = Path("../report/figures"); FIGS.mkdir(parents=True, exist_ok=True)  # report-ready
rng = np.random.default_rng(0)

def collect_raw_meta(split_name, limit=None):
    """Raw per-image properties BEFORE cleaning: (w, h, mode, format, nbytes)."""
    rows = []
    for i, (b, lab) in enumerate(_io.read_parquet_split(DATA / split_name)):
        if limit and i >= limit:
            break
        try:
            im = Image.open(_stdio.BytesIO(b))
            rows.append((im.width, im.height, im.mode, (im.format or "?"),
                         len(b), 0 if lab == 0 else 1))
        except Exception:
            rows.append((-1, -1, "ERR", "ERR", len(b), 0 if lab == 0 else 1))
    return rows

def decode_clean(split_name):
    """Cleaned 224x224 uint8 (N,224,224,3) + binary labels (cached)."""
    xp = NBCACHE / f"{split_name}_x.uint8.npy"
    yp = NBCACHE / f"{split_name}_y.npy"
    if xp.exists() and yp.exists():
        return np.load(xp), np.load(yp)
    imgs, labs = [], []
    for b, lab in _io.read_parquet_split(DATA / split_name):
        arr = _io.clean_image(b)
        if arr is None:
            continue
        imgs.append((arr * 255.0 + 0.5).astype(np.uint8))
        labs.append(0 if lab == 0 else 1)
    X = np.stack(imgs); yv = np.array(labs, dtype=np.int64)
    np.save(xp, X); np.save(yp, yv)
    return X, yv

SPLITS = ["calibration", "calibration_augmented", "validation", "validation_augmented"]
clean = {s: decode_clean(s) for s in SPLITS}
for s in SPLITS:
    X, y = clean[s]
    print(f"{s:24s} cleaned n={len(X):5d}  real={int((y==0).sum()):4d}  ai={int((y==1).sum()):4d}")


## 1 - Raw properties and pairing

Before looking at pixels: do the augmented splits differ in **encoded** form (image dimensions, aspect ratio, color mode, file format, byte size)? These reveal geometric resizing, format re-encoding, grayscale conversion, and compression that the cleaning step would otherwise mask. Then test whether `calibration_augmented` is row-aligned with `calibration` (same underlying image, augmented); if it is, we can compute exact paired deltas.

How to read the output: compare `calibration` vs `calibration_augmented` line by line. Same width/height/aspect/format/mode means no geometric or format change; a drop in median KB means heavier compression. A pairing diagonal near 1.0 would mean row-aligned; near 0.0 means not aligned.

In [ ]:
raw = {s: collect_raw_meta(s) for s in SPLITS}

def raw_summary(s):
    rows = raw[s]
    w = np.array([r[0] for r in rows]); h = np.array([r[1] for r in rows])
    nb = np.array([r[4] for r in rows]); modes = [r[2] for r in rows]; fmts = [r[3] for r in rows]
    ar = w / np.maximum(h, 1)
    from collections import Counter
    print(f"\n[{s}] n={len(rows)}")
    print(f"  width : min={w.min()} med={int(np.median(w))} max={w.max()}")
    print(f"  height: min={h.min()} med={int(np.median(h))} max={h.max()}")
    print(f"  aspect: min={ar.min():.3f} med={np.median(ar):.3f} max={ar.max():.3f}  square(==1)={int((w==h).sum())}/{len(rows)}")
    print(f"  bytes : med={int(np.median(nb))}  (KB med={np.median(nb)/1024:.1f})")
    print(f"  modes : {dict(Counter(modes))}")
    print(f"  format: {dict(Counter(fmts))}")

for s in SPLITS:
    raw_summary(s)

# pairing check: calibration vs calibration_augmented (cleaned thumbnails)
def thumbs(X, n=16):
    g = (0.299*X[...,0] + 0.587*X[...,1] + 0.114*X[...,2]).astype(np.float32)
    # block-average to n x n
    f = 224 // n
    g = g[:, :f*n, :f*n].reshape(len(g), n, f, n, f).mean(axis=(2,4))
    return g.reshape(len(g), -1)

Xc, yc = clean["calibration"]; Xca, yca = clean["calibration_augmented"]
print("\npairing check (calibration vs calibration_augmented):")
if len(Xc) == len(Xca):
    print(f"  per-index label match: {(yc==yca).mean():.3f}")
    smp = np.sort(rng.choice(len(Xca), size=min(200, len(Xca)), replace=False))
    Tc = thumbs(Xc); Tca = thumbs(Xca[smp])
    # nearest clean thumbnail for each sampled augmented thumbnail
    d = ((Tca[:, None, :] - Tc[None, :, :])**2).sum(-1)
    nn = d.argmin(1)
    diag = (nn == smp).mean()
    print(f"  nearest-clean-thumbnail lands on same index: {diag:.3f} (1.0 => row-aligned)")
else:
    print(f"  different lengths ({len(Xc)} vs {len(Xca)}) -> not row-aligned")


## 2 - Pixel-space metric battery

Compare clean vs augmented (per class) on sharpness (Laplacian variance), high-frequency spectral power, noise-residual std, per-channel brightness, within-image contrast, and saturation. The direction of each delta narrows the transform family. Splits are not row-aligned, so these are distribution means, not paired diffs.

In [ ]:
def _gray(X):
    Xf = X.astype(np.float32) / 255.0
    return 0.299*Xf[...,0] + 0.587*Xf[...,1] + 0.114*Xf[...,2]

def _chunked(fn, X, cs=200):
    """Apply a per-image metric fn in chunks to bound peak memory (FFT is heavy)."""
    out = []
    for i in range(0, len(X), cs):
        out.append(fn(X[i:i+cs]))
    return np.concatenate(out) if out else np.array([])

def _lapvar(X):
    g = _gray(X)
    lap = (-4*g[:,1:-1,1:-1] + g[:,:-2,1:-1] + g[:,2:,1:-1] + g[:,1:-1,:-2] + g[:,1:-1,2:])
    return lap.reshape(len(g), -1).var(1)

_FFTMASK = (np.sqrt(np.add.outer(np.fft.fftfreq(224)**2, np.fft.fftfreq(224)**2)) > 0.25)
def _hf(X):
    g = _gray(X)
    sp = np.abs(np.fft.fft2(g, axes=(1,2)))**2
    return sp[:, _FFTMASK].sum(1) / (sp.reshape(len(g), -1).sum(1) + 1e-9)

def _noise(X):
    g = _gray(X)
    bl = (g[:,:-2,:-2]+g[:,:-2,1:-1]+g[:,:-2,2:]+g[:,1:-1,:-2]+g[:,1:-1,1:-1]
          +g[:,1:-1,2:]+g[:,2:,:-2]+g[:,2:,1:-1]+g[:,2:,2:]) / 9.0
    return (g[:,1:-1,1:-1] - bl).reshape(len(g), -1).std(1)

# public chunked wrappers used throughout the notebook
def m_lapvar(X): return _chunked(_lapvar, X)
def m_hf(X):     return _chunked(_hf, X)
def m_noise(X):  return _chunked(_noise, X)

def m_color(X):
    Xf = X.astype(np.float32) / 255.0
    bright = Xf.mean((1,2)); contrast = Xf.std((1,2))
    mx = Xf.max(3); mn = Xf.min(3)
    sat = ((mx - mn) / (mx + 1e-6)).reshape(len(X), -1).mean(1)
    return bright, contrast, sat

def battery(tag, X, y):
    for c, nm in [(0, "real"), (1, "ai")]:
        m = y == c
        if m.sum() == 0: continue
        Xm = X[m]
        b, ct, sat = m_color(Xm)
        print(f"  {tag:22s} {nm:4s} n={int(m.sum()):4d}  lapvar={m_lapvar(Xm).mean():.5f}  "
              f"hf={m_hf(Xm).mean():.5f}  noise={m_noise(Xm).mean():.5f}  "
              f"bright={b.mean():.3f}  contrast={ct.mean():.3f}  sat={sat.mean():.3f}")

print("metric battery (cleaned 224 space, distribution means):")
battery("calibration",           clean["calibration"][0],           clean["calibration"][1])
battery("calibration_augmented", clean["calibration_augmented"][0], clean["calibration_augmented"][1])
battery("validation",            clean["validation"][0],            clean["validation"][1])
battery("validation_augmented",  clean["validation_augmented"][0],  clean["validation_augmented"][1])


## 5 - Direct JPEG quality (quantization tables)

Rather than inferring compression from byte size, read the luminance quantization table straight from each JPEG (`PIL.Image.quantization`) and estimate its quality factor. Larger quant values = heavier compression. This is a direct measurement of the re-encode, not an inference. (The estimator saturates near max quality, so it resolves the *low-quality tail* better than the high end.)

In [ ]:
# standard JPEG luminance quantization table (Annex K)
STD_LUM = np.array([16,11,10,16,24,40,51,61, 12,12,14,19,26,58,60,55,
                    14,13,16,24,40,57,69,56, 14,17,22,29,51,87,80,62,
                    18,22,37,56,68,109,103,77, 24,35,55,64,81,104,113,92,
                    49,64,78,87,103,121,120,101, 72,92,95,98,112,100,103,99], dtype=float)

def est_quality(qtable):
    q = np.array(qtable, dtype=float)
    scale = np.median(q / STD_LUM) * 100.0
    if scale <= 0: return 100.0
    Q = (200 - scale) / 2 if scale >= 100 else 5000.0 / scale
    return float(np.clip(Q, 1, 100))

def quality_stats(split, limit=600):
    qs, qmean = [], []
    for i, (b, lab) in enumerate(_io.read_parquet_split(DATA / split)):
        im = Image.open(_stdio.BytesIO(b))
        qt = im.quantization
        if qt and 0 in qt:
            qs.append(est_quality(qt[0])); qmean.append(np.mean(qt[0]))
        if i + 1 >= limit: break
    qs = np.array(qs); qmean = np.array(qmean)
    print(f"  {split:24s} n={len(qs):4d}  estQ p10={np.percentile(qs,10):5.0f} med={np.median(qs):5.0f} "
          f"min={qs.min():4.0f}  qtable_mean med={np.median(qmean):5.1f}  frac(estQ<90)={ (qs<90).mean():.2f}")

print("JPEG quality from quantization tables:")
for s in SPLITS:
    quality_stats(s)


## 6 - Distributions and per-image randomness

Means hide whether *every* image is degraded or only a *fraction*. Histograms of `hf_ratio` and noise residual (clean vs augmented, reals) plus the fraction of augmented images still near clean tell us whether the dataset's augmentation is applied per image with a probability and variable strength.

What the figure shows: two overlaid histograms (clean vs augmented). If the augmented histogram were a single shifted peak, all images were degraded uniformly; instead a chunk of augmented mass overlaps the clean distribution, meaning a fraction of images are barely touched (per-image random application). The printed "fraction still near clean" quantifies this. Saved to `report/figures/`.

In [ ]:
Xc, yc = clean["calibration"]; Xca, yca = clean["calibration_augmented"]
hc = m_hf(Xc[yc==0]); ha = m_hf(Xca[yca==0])
nc = m_noise(Xc[yc==0]); na = m_noise(Xca[yca==0])

p25_clean = np.percentile(hc, 25)
frac_clean = (ha >= p25_clean).mean()
print(f"hf_ratio (reals):  clean med={np.median(hc):.4f} p90={np.percentile(hc,90):.4f}  "
      f"aug med={np.median(ha):.4f} p90={np.percentile(ha,90):.4f}")
print(f"fraction of augmented reals still >= clean p25 ({p25_clean:.4f}): {frac_clean:.2f}")
print("=> a substantial near-clean fraction implies per-image random application, not uniform.")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.linspace(0, 0.02, 50)
ax[0].hist(hc, bins=bins, alpha=0.6, label="clean", density=True)
ax[0].hist(ha, bins=bins, alpha=0.6, label="augmented", density=True)
ax[0].set_title("hf_ratio (reals)"); ax[0].set_xlabel("hf_ratio"); ax[0].legend()
bins2 = np.linspace(0, 0.06, 50)
ax[1].hist(nc, bins=bins2, alpha=0.6, label="clean", density=True)
ax[1].hist(na, bins=bins2, alpha=0.6, label="augmented", density=True)
ax[1].set_title("noise residual (reals)"); ax[1].set_xlabel("noise std"); ax[1].legend()
fig.tight_layout(); fig.savefig(FIGS / "distributions_real.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved", FIGS / "distributions_real.png")


## 3 - Visual comparison (provided augmented data)

These are real images from the dataset's `*_augmented` split; we did not augment them. The provided augmented split is **not** the same photos as the clean split: nearest-neighbor matching on contrast-normalized thumbnails (robust to blur and JPEG) finds a confident clean original for only a minority of augmented reals, and those few are the near-unchanged ones. So the same provided photo cannot be shown clean vs augmented for the visibly-degraded cases.

What the figure shows: top row is random clean reals; bottom row is the **most-degraded** provided augmented reals (lowest high-frequency power), chosen so the augmentation is actually visible. They are unpaired (different photos). Look for softening, JPEG blockiness, and lower saturation, not geometric or color changes. (The before/after of our own augmentation lives in notebook 08.) Saved to `report/figures/`.

In [ ]:
Xc, yc = clean["calibration"]; Xca, yca = clean["calibration_augmented"]

# quantify overlap: can we find each augmented real's clean original? (robust thumbnail NN)
def _thumb(X, n=32):
    g = (0.299*X[...,0] + 0.587*X[...,1] + 0.114*X[...,2]).astype(np.float32)
    f = 224 // n; g = g[:, :f*n, :f*n].reshape(len(g), n, f, n, f).mean((2,4)).reshape(len(g), -1)
    return (g - g.mean(1, keepdims=True)) / (g.std(1, keepdims=True) + 1e-6)
cr = Xc[yc==0]; ar = Xca[yca==0]
Dc = _thumb(cr); Da = _thumb(ar)
nn_d = np.array([((Dc - Da[i])**2).sum(1).min() for i in range(len(Da))])
print(f"provided augmented reals with a confident clean match (NN dist < 20): "
      f"{int((nn_d < 20).sum())}/{len(ar)}  (splits mostly contain different photos)")

def show_examples(cls, name, fname):
    Xc_c = clean["calibration"][0][clean["calibration"][1] == cls]
    Xa_c = clean["calibration_augmented"][0][clean["calibration_augmented"][1] == cls]
    worst = np.argsort(m_hf(Xa_c))[:6]                  # most-degraded provided augmented
    ci = rng.choice(len(Xc_c), 6, replace=False)        # random clean
    fig, ax = plt.subplots(2, 6, figsize=(12, 4.4))
    for j in range(6):
        ax[0, j].imshow(Xc_c[ci[j]]);    ax[0, j].axis("off")
        ax[1, j].imshow(Xa_c[worst[j]]); ax[1, j].axis("off")
    fig.suptitle(f"{name}: random clean (top) vs most-degraded provided augmented (bottom), unpaired")
    fig.tight_layout(); fig.savefig(FIGS / fname, dpi=110, bbox_inches="tight")
    plt.show()

show_examples(0, "real", "provided_aug_examples_real.png")
show_examples(1, "ai",   "provided_aug_examples_ai.png")
print("saved provided-augmented example grids to", FIGS)


## 7 - Conclusions (cautious)

Evidence is observational and from **unpaired** splits, so we identify the transform *family*, its per-image application, and rough magnitude, not the exact generator. Scope note: all raw-property analysis is on the provided **320x320** JPEG encodings; our pipeline then cleans every image to a fixed **224x224** canvas (Task 1.1, matches the reference CNN), which is the working size for the pixel metrics and for training.

**Ruled in (multiple independent signals agree):** a per-image, randomly-applied low-pass / quality-reduction shift.
- Encoded byte size roughly halves (cal 30.5 to 17.5 KB; val 30.8 to 18.2 KB) at unchanged 320x320 JPEG: re-compression.
- Quantization tables (direct): augmented JPEGs have a heavy low-quality tail (estQ p10 ~14 to 22, min ~1) absent in clean; the median stays high, so only a *subset* is strongly compressed.
- High-frequency spectral power and Laplacian variance drop, power shifts to the low band, noise residual drops: blur and/or downsample.
- Saturation drops ~20%, contrast ~10%: consistent with JPEG chroma subsampling plus smoothing.
- Distribution check: ~43% of augmented reals stay near clean (hf >= clean p25) while the rest are degraded with varying strength, so augmentation is applied **per image with a probability and random strength**, not uniformly.
- Forward-matching: no single transform matches all metrics; the augmented point sits between moderate JPEG and mild blur/downsample, i.e. a **combination**.

**No positive evidence found (absence of evidence is not proof it was not applied):**
- Crop / aspect change / resize to a different size: encoded dims are identical 320x320 in clean and augmented, so these are unlikely. Important caveat: a downscale-then-upscale resampling preserves the 320x320 dims, so it is **not** excluded; it is part of our suspected blur/quality family.
- Format or grayscale conversion: still JPEG, still RGB (high confidence none applied).
- Global brightness / color cast: per-channel means barely change, so a strong global shift is unlikely; mild per-image brightness/contrast jitter could be hidden in the average.
- Additive noise: the noise residual *decreased*, which is evidence against a *net* noise increase; a small amount of noise combined with stronger blur cannot be ruled out.

**Cannot determine from unpaired data (so not claimed):** horizontal flip and 90-degree rotation preserve every metric we measured; exact JPEG quality / blur sigma / downsample factor and their order are not identifiable. Operation order matters (JPEG at 320 then downsample is not the same as JPEG at 224), which is why the 224-space match overstates JPEG strength vs the native byte-size and quant-table views.

**Implied augmentation candidate for notebook 08** (a starting point; the actual choice is decided there by ablation/runs):
- Per-image random application (probability plus random strength), matching the observed mixture.
- Primary (cover the shift): random JPEG re-encode (quality mostly ~[45, 90], occasional heavy ~[12, 40]); random Gaussian blur (sigma ~[0.3, 0.9]); random downscale-upscale (factor ~[0.7, 0.92]).
- Low-risk extras: horizontal flip (cannot rule out, label-preserving, applied on the fly during training only); very mild brightness/contrast jitter.
- Lower priority / omit unless ablation shows benefit: additive noise (shift goes the other way); geometric crops that could re-introduce a dimension signal; hue / heavy color jitter (no evidence).

**Mimic vs general robustness (decision):** we cover this observed family with somewhat broader randomized ranges rather than precision-fitting the exact provided parameters. Precision-fitting risks overfitting to the one known shift; the PDF rewards general robustness and improving beyond this level. We also never tune augmentation strength on `validation_augmented` (held-out test); selection uses a train-derived augmented holdout and the threshold uses `calibration_augmented`. See the experiment log for the full rationale.

Note: augmentation is applied on the fly during training, never stored as a new split. The candidate augmentation, its visuals, and the comparison against this provided shift are built and decided in notebook 08.

Figures saved to `report/figures/`: `provided_aug_examples_real.png`, `provided_aug_examples_ai.png`, `distributions_real.png`. Findings recorded in `task13_experiment_log.md`.

## Port targets

- Findings table + figure list -> `task13_experiment_log.md` (Provided-shift findings).
- Figures (`provided_aug_examples_real.png`, `provided_aug_examples_ai.png`, `distributions_real.png`) -> `report/figures/`, cited in report `report.md` §1.3.
- The implied augmentation candidate (per-image random JPEG re-encode + mild blur/downscale, optional desaturation) is built and validated in `08_task13_augmentation.ipynb`.